# Image Captioning Pipeline — CNN-RNN on MS COCO
## Advanced Machine Learning — University Project

| Field         | Details              |
|---------------|----------------------|
| Student 1     | Ahmed                |
| Student 1 ID  | 13004667     |
| Student 2     | Mazen             |
| Student 2 ID  | 14005132              |
| Student 3     | Hassan Eslam             |
| Student 3 ID  | 13001118              |

**Architecture:** CNN Encoder (ResNet50) + LSTM Decoder  
**Dataset:** MS COCO 2017 (>=50% subset, seed=42)

In [1]:
# Basic libraries
import os
import json
import re
import string
import pickle
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# Text preprocessing (Keras utilities — work fine alongside PyTorch models)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# NOTE: InceptionV3/Keras CNN imports removed — actual model uses PyTorch ResNet50

# File paths for MS COCO (Google Colab paths)
CAPTIONS_FILE = '/content/annotations/captions_train2017.json'
IMAGES_DIR = '/content/train2017'

# Files to save preprocessing outputs
FEATURES_FILE = '/content/coco_image_features.pkl'
TOKENIZER_FILE = '/content/tokenizer.pkl'
MAPPINGS_FILE = '/content/vocab_mappings.pkl'

In [2]:
import random

# Load MS COCO captions JSON
with open(CAPTIONS_FILE, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# ── SUBSET SAMPLING (satisfies >=50% requirement) ──────────────────────────
random.seed(42)  # Fixed seed ensures the same subset every run (reproducibility)
all_image_ids = [img['id'] for img in coco_data['images']]
sample_size   = int(len(all_image_ids) * 0.60)  # 60% > minimum 50% threshold
sampled_ids   = set(random.sample(all_image_ids, sample_size))

print(f'Total images in MS COCO 2017 train: {len(all_image_ids)}')
print(f'Sampled subset : {len(sampled_ids)} images ({len(sampled_ids)/len(all_image_ids)*100:.1f}%)')

# Build image_id -> file_name mapping for sampled images only
image_id_to_file = {
    img['id']: img['file_name']
    for img in coco_data['images']
    if img['id'] in sampled_ids
}

# Collect captions grouped by file name (sampled images only)
captions_mapping = {}
for ann in coco_data['annotations']:
    image_id = ann['image_id']
    if image_id not in sampled_ids:  # skip images outside our subset
        continue
    file_name = image_id_to_file[image_id]
    caption   = ann['caption']
    captions_mapping.setdefault(file_name, []).append(caption)

print('Number of sampled images with captions:', len(captions_mapping))
sample_file = next(iter(captions_mapping))
print('Sample image:', sample_file)
print('Sample captions:', captions_mapping[sample_file][:2])

FileNotFoundError: [Errno 2] No such file or directory: '/content/annotations/captions_train2017.json'

In [ ]:
# Clean each caption in a simple standard way
def clean_caption(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep letters and spaces only
    text = re.sub(r'\s+', ' ', text).strip()
    words = [word for word in text.split() if len(word) > 1]
    return ' '.join(words)

# Add start/end tokens for sequence generation
cleaned_captions_mapping = {}
all_cleaned_captions = []

for file_name, captions in captions_mapping.items():
    cleaned_list = []
    for caption in captions:
        cleaned = clean_caption(caption)
        cleaned = '<start> ' + cleaned + ' <end>'
        cleaned_list.append(cleaned)
        all_cleaned_captions.append(cleaned)
    cleaned_captions_mapping[file_name] = cleaned_list

print('Total cleaned captions:', len(all_cleaned_captions))
print('Example cleaned caption:', all_cleaned_captions[0])


In [ ]:
# Tokenize processed captions
tokenizer = Tokenizer(oov_token='<unk>', filters='')
tokenizer.fit_on_texts(all_cleaned_captions)

# Convert captions into token sequences
caption_sequences = tokenizer.texts_to_sequences(all_cleaned_captions)

print('Vocabulary size from tokenizer:', len(tokenizer.word_index) + 1)
print('Sample tokenized caption:', caption_sequences[0])


In [ ]:
# === TASK 2: VOCABULARY CONSTRUCTION ===
# Build vocabulary with frequency filtering

# Count word frequencies
word_counts = Counter()
for caption in all_cleaned_captions:
    word_counts.update(caption.split())

# Apply frequency threshold (keep words appearing >= 5 times)
MIN_FREQUENCY = 5
frequent_words = {word: count for word, count in word_counts.items() if count >= MIN_FREQUENCY}

# Tokenize with filtered vocabulary
tokenizer = Tokenizer(num_words=len(frequent_words) + 1, oov_token='<unk>', filters='')
tokenizer.fit_on_texts(all_cleaned_captions)

# Build explicit mappings (CRITICAL: index 0 is padding)
word_to_index = {'<pad>': 0}  # Reserve index 0 for padding
word_to_index.update(tokenizer.word_index)  # Add tokenizer indices

index_to_word = {idx: word for word, idx in word_to_index.items()}
vocab_size = len(word_to_index)

# Verify special tokens
assert '<start>' in word_to_index, "ERROR: <start> not in vocabulary!"
assert '<end>' in word_to_index, "ERROR: <end> not in vocabulary!"

print('Vocabulary size:', vocab_size)
print('Index of <pad>:', word_to_index.get('<pad>'))
print('Index of <start>:', word_to_index.get('<start>'))
print('Index of <end>:', word_to_index.get('<end>'))

In [ ]:
# Save tokenizer and mappings for later use
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)

with open(MAPPINGS_FILE, 'wb') as f:
    pickle.dump({
        'word_to_index': word_to_index,
        'index_to_word': index_to_word,
        'vocab_size': vocab_size
    }, f)

print('Tokenizer and vocabulary mappings saved.')


In [ ]:
# Recompute sequences using the FINAL filtered tokenizer (Tokenizer #2 from the vocab cell).
# The previous caption_sequences came from Tokenizer #1 (no frequency filter, now discarded).
# Using a stale tokenizer for max_caption_length would be semantically incorrect.
caption_sequences = tokenizer.texts_to_sequences(all_cleaned_captions)

# Find the maximum sequence length — used as maxlen in pad_sequences
max_caption_length = max(len(seq) for seq in caption_sequences)
print('Maximum caption length:', max_caption_length)

In [ ]:
# Prepare input-output sequences for caption generation preprocessing
# X_text: partial caption so far
# y_word: next word to predict
# X_image_names: matching image file name for each text sequence

X_text = []
y_word = []
X_image_names = []

for file_name, captions in cleaned_captions_mapping.items():
    for caption in captions:
        seq = tokenizer.texts_to_sequences([caption])[0]

        # Create many training pairs from one caption
        for i in range(1, len(seq)):
            in_seq = seq[:i]
            out_seq = seq[i]
            X_text.append(in_seq)
            y_word.append(out_seq)
            X_image_names.append(file_name)

# Pad input text sequences to same length
X_text = pad_sequences(X_text, maxlen=max_caption_length, padding='post')
y_word = np.array(y_word)
X_image_names = np.array(X_image_names)

print('Text input shape:', X_text.shape)
print('Target output shape:', y_word.shape)
print('Image reference shape:', X_image_names.shape)


In [ ]:
# Show one prepared example
example_index = 0
print('Image file:', X_image_names[example_index])
print('Padded input sequence:', X_text[example_index])
print('Target next word index:', y_word[example_index])
print('Target next word:', index_to_word.get(int(y_word[example_index]), '<unknown>'))


In [ ]:
# === TASK 4: IMAGE FEATURE EXTRACTION (CNN) ===
# Extract features using ResNet50 (consistent with PyTorch model)

import torch
import torchvision.models as models
from torchvision import transforms

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load pretrained ResNet50
resnet50 = models.resnet50(pretrained=True)

# Remove classification layer - keep feature extractor
modules = list(resnet50.children())[:-1]
feature_extractor = torch.nn.Sequential(*modules)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Freeze parameters (transfer learning - no fine-tuning)
for param in feature_extractor.parameters():
    param.requires_grad = False

print('CNN model ready (ResNet50)')
print('Output feature dimension: 2048')

# Preprocessing pipeline for ImageNet
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [ ]:
# Function to extract one image feature vector
def extract_image_feature(img_path, model, preprocessing, device):
    from PIL import Image
    try:
        img = Image.open(img_path).convert('RGB')
        img_tensor = preprocessing(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            features = model(img_tensor)
        
        return features.flatten().cpu().numpy()
    except:
        return None

In [ ]:
# Extract and store one feature vector per image
image_features = {}
failed_count = 0

for file_name in tqdm(cleaned_captions_mapping.keys()):
    img_path = os.path.join(IMAGES_DIR, file_name)
    if os.path.exists(img_path):
        feature = extract_image_feature(img_path, feature_extractor, preprocess, device)
        if feature is not None:
            image_features[file_name] = feature
        else:
            failed_count += 1
    else:
        failed_count += 1

print('Extracted features for images:', len(image_features))
print('Failed/missing images:', failed_count)

# Remove images without features from training set
for img in list(cleaned_captions_mapping.keys()):
    if img not in image_features:
        del cleaned_captions_mapping[img]

In [ ]:
# Save extracted feature vectors
with open(FEATURES_FILE, 'wb') as f:
    pickle.dump(image_features, f)

print('Image features saved to:', FEATURES_FILE)


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()

        # Load pre-trained ResNet50 (weights learned from 1.2M ImageNet images)
        resnet = models.resnet50(pretrained=True)

        # Remove the final classification layer; keep only the feature extractor
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # FIX: Freeze all backbone weights inside the class.
        # Without this, the entire 25M-parameter ResNet updates every training step,
        # destroying pre-trained ImageNet features and wasting GPU memory.
        for param in self.resnet.parameters():
            param.requires_grad = False

        # Linear projection: 2048-dim ResNet output -> embed_size
        # This is the only encoder layer that will actually be trained.
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)

        # BatchNorm stabilises projected features before they enter the LSTM
        self.batch_norm = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        # Frozen ResNet forward pass — no gradients computed here
        features = self.resnet(images)

        # Flatten: (batch, 2048, 1, 1) -> (batch, 2048)
        features = features.view(features.size(0), -1)

        # Project to embedding space and normalise
        features = self.embed(features)
        features = self.batch_norm(features)

        return features


In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, dropout=0.5):
        super(DecoderRNN, self).__init__()

        # (b) Embedding Layer
        # padding_idx=0: <pad> token always produces a zero vector (no gradient)
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        nn.init.uniform_(self.embed.weight, -0.05, 0.05)
        self.embed_norm = nn.LayerNorm(embed_size)

        # FIX: Explicit Dropout layer.
        # nn.LSTM dropout only fires BETWEEN layers (useless when num_layers=1).
        # nn.Dropout works for any layer count and auto-disables during model.eval().
        self.dropout = nn.Dropout(p=dropout)

        # (c) RNN Decoder (LSTM)
        self.lstm = nn.LSTM(
            embed_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0  # inter-layer dropout if >1 layers
        )

        # (d) Final Dense Layer — projects LSTM output to vocabulary logits
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        # Strip <end> token — model is not asked to predict beyond sequence end
        captions = captions[:, :-1]

        # Embed caption tokens -> (batch, seq_len, embed_size)
        embeddings = self.embed(captions)
        embeddings = self.embed_norm(embeddings)

        # Apply dropout to embeddings before the LSTM
        embeddings = self.dropout(embeddings)

        # Prepend image feature vector as the first input token
        features = features.unsqueeze(1)  # (batch, 1, embed_size)

        # Concatenate along sequence dimension: [img_feat, word1, word2, ...]
        inputs = torch.cat((features, embeddings), dim=1)

        # Run through LSTM
        lstm_out, _ = self.lstm(inputs)

        # Apply dropout to LSTM output before the final projection
        lstm_out = self.dropout(lstm_out)

        # Project to vocabulary logits -> (batch, seq_len, vocab_size)
        outputs = self.linear(lstm_out)

        return outputs


In [ ]:
# Hyperparameters (Example values)
embed_size = 256
hidden_size = 512
vocab_size = 10000 # This should match the length of the vocabulary you built in step (b)
num_layers = 1

# Initialize the Encoder (CNN) and Decoder (RNN)
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

# Example Forward Pass logic (used during your training loop)
# images: (batch_size, 3, 224, 224) 
# captions: (batch_size, max_seq_length)

# 1. Extract visual features
# features = encoder(images)

# 2. Generate caption predictions
# outputs = decoder(features, captions)

In [ ]:
# === TRAINING LOOP ===
import torch.optim as optim

# ── Hyperparameters ───────────────────────────────────────────────────────
EMBED_SIZE    = 256
HIDDEN_SIZE   = 512
NUM_LAYERS    = 1
DROPOUT       = 0.5
BATCH_SIZE    = 64
NUM_EPOCHS    = 5
LEARNING_RATE = 3e-4

# ── Re-initialise models with correct vocab size ──────────────────────────
encoder = EncoderCNN(EMBED_SIZE).to(device)
decoder = DecoderRNN(EMBED_SIZE, HIDDEN_SIZE, vocab_size, NUM_LAYERS, DROPOUT).to(device)

# ── Loss Function ─────────────────────────────────────────────────────────
# CrossEntropyLoss: standard for next-word prediction (multi-class classification).
# ignore_index=0 skips <pad> positions so they don't distort the loss.
criterion = nn.CrossEntropyLoss(ignore_index=0)

# ── Optimizer ─────────────────────────────────────────────────────────────
# Only the decoder + encoder projection layers have requires_grad=True.
# The frozen ResNet backbone is automatically excluded from updates.
params = (
    list(decoder.parameters())
    + list(encoder.embed.parameters())
    + list(encoder.batch_norm.parameters())
)
optimizer = optim.Adam(params, lr=LEARNING_RATE)

# ── Training Mode ─────────────────────────────────────────────────────────
# .train() activates Dropout and updates BatchNorm running statistics.
# .eval()  disables Dropout and freezes BatchNorm (use during inference).
encoder.train()
decoder.train()

# ── Training Loop Skeleton ────────────────────────────────────────────────
# NOTE: Replace `train_dataloader` with a proper DataLoader wrapping a
# torch.utils.data.Dataset that yields (image_tensor, caption_tensor) pairs.
for epoch in range(NUM_EPOCHS):
    total_loss  = 0.0
    num_batches = 0

    for batch_images, batch_captions in train_dataloader:  # replace with your DataLoader
        batch_images   = batch_images.to(device)    # (batch, 3, 224, 224)
        batch_captions = batch_captions.to(device)  # (batch, max_seq_len)

        # ── Forward Pass ──
        features = encoder(batch_images)              # (batch, embed_size)
        outputs  = decoder(features, batch_captions)  # (batch, seq_len, vocab_size)

        # Targets: captions shifted left by 1 (next-word prediction)
        targets = batch_captions[:, 1:]  # strip <start> token

        # CrossEntropyLoss needs (N, C) and (N,) shapes
        loss = criterion(
            outputs.reshape(-1, vocab_size),  # (batch*seq_len, vocab_size)
            targets.reshape(-1)               # (batch*seq_len,)
        )

        # ── Backward Pass ──
        optimizer.zero_grad()                         # clear old gradients
        loss.backward()                               # compute new gradients
        nn.utils.clip_grad_norm_(params, max_norm=5.0)  # prevent gradient explosion in RNNs
        optimizer.step()                              # update weights

        total_loss  += loss.item()
        num_batches += 1

    avg_loss = total_loss / num_batches
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  Avg Loss: {avg_loss:.4f}')
